<a href="https://colab.research.google.com/github/shini-080507/insurance-/blob/main/Insurance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib seaborn openai pypdf pytesseract pdf2image pillow gradio

import pandas as pd
import numpy as np
import json
import os

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

from pypdf import PdfReader
from PIL import Image
import pytesseract

from openai import OpenAI
from google.colab import files

data = {
    "age": [25,45,32,56,40,29,61,35,48,52,
            27,43,67,38,50,31,59,42,36,64],

    "hospitalization_days": [2,7,3,12,8,1,10,4,6,9,
                             2,5,15,3,7,2,11,6,4,13],

    "medical_expenses": [15000,85000,25000,150000,95000,
                         10000,180000,35000,70000,110000,
                         18000,60000,220000,30000,90000,
                         20000,160000,75000,45000,200000],

    "previous_claims": [0,2,1,3,2,0,4,1,2,3,
                        0,1,5,1,2,0,3,2,1,4],

    "severity": [
        "Low","High","Low","High","Medium",
        "Low","High","Low","Medium","High",
        "Low","Medium","High","Low","Medium",
        "Low","High","Medium","Low","High"
    ]
}

df = pd.DataFrame(data)

df.head()

severity_mapping = {
    "Low": 0,
    "Medium": 1,
    "High": 2
}

reverse_mapping = {
    0: "Low",
    1: "Medium",
    2: "High"
}

df["severity_encoded"] = df["severity"].map(severity_mapping)

features = [
    "age",
    "hospitalization_days",
    "medical_expenses",
    "previous_claims"
]

X = df[features]
y = df["severity_encoded"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Data prepared successfully.")

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Random Forest Accuracy:", round(accuracy * 100, 2), "%")
print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Low", "Medium", "High"],
    zero_division=0
))

importance = pd.DataFrame({
    "Feature": features,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False)

display(importance)

plt.figure(figsize=(8,5))
plt.barh(importance["Feature"], importance["Importance"])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Feature Importance")
plt.gca().invert_yaxis()
plt.show()

def predict_claim_severity(
    age,
    hospitalization_days,
    medical_expenses,
    previous_claims
):
    claim = pd.DataFrame({
        "age": [age],
        "hospitalization_days": [hospitalization_days],
        "medical_expenses": [medical_expenses],
        "previous_claims": [previous_claims]
    })

    prediction = model.predict(claim)[0]

    return reverse_mapping[prediction]


# Example
print(
    predict_claim_severity(
        45,
        7,
        85000,
        2
    )
)

from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass(
    "Enter your OpenAI API key: "
)

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"]
)

print("OpenAI client configured.")

def parse_medical_bill(bill_text):

    prompt = f"""
Extract information from this medical bill.

Return ONLY valid JSON.

Required fields:

patient_name
hospital_name
policy_number
treatment
hospitalization_days
room_charges
surgery_charges
doctor_charges
medicine_charges
diagnostic_charges
non_medical_expenses
total_bill_amount

Use 0 for missing numerical values.

Medical Bill:
{bill_text}
"""

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {
                "role": "system",
                "content": "You are a medical bill document parser. Return only valid JSON."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return json.loads(
        response.choices[0].message.content
    )

def extract_bill_text(filename):

    filename = filename.lower()

    if filename.endswith(".pdf"):

        reader = PdfReader(filename)

        text = ""

        for page in reader.pages:
            page_text = page.extract_text()

            if page_text:
                text += page_text + "\n"

        return text

    elif filename.endswith((".png", ".jpg", ".jpeg")):

        image = Image.open(filename)

        return pytesseract.image_to_string(image)

    elif filename.endswith(".txt"):

        with open(filename, "r", encoding="utf-8") as file:
            return file.read()

    else:
        raise ValueError(
            "Only PDF, PNG, JPG, JPEG and TXT files are supported."
        )

policy_rules = {
    "room_charge_limit": 5000,
    "surgery_coverage": True,
    "doctor_fee_coverage": True,
    "medicine_coverage": True,
    "diagnostic_coverage": True,
    "non_medical_expenses": False,
    "maximum_coverage": 100000
}

policy_rules

def verify_policy(bill, policy):

    covered = 0
    non_covered = 0

    covered_items = []
    rejected_items = []

    # Room charges
    room = float(bill.get("room_charges", 0))
    days = int(bill.get("hospitalization_days", 0))

    allowed_room = policy["room_charge_limit"] * days

    if room <= allowed_room:
        covered += room
        covered_items.append("Room charges")
    else:
        covered += allowed_room
        non_covered += room - allowed_room
        rejected_items.append("Room charges above limit")

    # Surgery
    surgery = float(bill.get("surgery_charges", 0))

    if policy["surgery_coverage"]:
        covered += surgery
        covered_items.append("Surgery charges")
    else:
        non_covered += surgery
        rejected_items.append("Surgery charges")

    # Doctor
    doctor = float(bill.get("doctor_charges", 0))

    if policy["doctor_fee_coverage"]:
        covered += doctor
        covered_items.append("Doctor charges")
    else:
        non_covered += doctor
        rejected_items.append("Doctor charges")

    # Medicine
    medicine = float(bill.get("medicine_charges", 0))

    if policy["medicine_coverage"]:
        covered += medicine
        covered_items.append("Medicine charges")
    else:
        non_covered += medicine
        rejected_items.append("Medicine charges")

    # Diagnostics
    diagnostic = float(bill.get("diagnostic_charges", 0))

    if policy["diagnostic_coverage"]:
        covered += diagnostic
        covered_items.append("Diagnostic charges")
    else:
        non_covered += diagnostic
        rejected_items.append("Diagnostic charges")

    # Non-medical
    non_medical = float(
        bill.get("non_medical_expenses", 0)
    )

    if policy["non_medical_expenses"]:
        covered += non_medical
        covered_items.append("Non-medical expenses")
    else:
        non_covered += non_medical
        rejected_items.append("Non-medical expenses")

    # Maximum coverage
    if covered > policy["maximum_coverage"]:

        excess = covered - policy["maximum_coverage"]

        covered = policy["maximum_coverage"]

        non_covered += excess

        rejected_items.append(
            "Amount exceeding maximum coverage"
        )

    if non_covered == 0:
        status = "Fully Covered"

    elif covered > 0:
        status = "Partially Covered"

    else:
        status = "Not Covered"

    return {
        "covered_amount": covered,
        "non_covered_amount": non_covered,
        "verification_status": status,
        "covered_items": covered_items,
        "rejected_items": rejected_items
    }

def analyze_claim(
    claim_id,
    age,
    hospitalization_days,
    medical_expenses,
    previous_claims,
    bill_filename
):

    # ML prediction
    severity = predict_claim_severity(
        age,
        hospitalization_days,
        medical_expenses,
        previous_claims
    )

    # Extract bill text
    bill_text = extract_bill_text(
        bill_filename
    )

    # LLM extraction
    bill_data = parse_medical_bill(
        bill_text
    )

    # Policy verification
    verification = verify_policy(
        bill_data,
        policy_rules
    )

    # Final result
    result = {
        "claim_id": claim_id,
        "claim_severity": severity,
        "patient_name": bill_data.get("patient_name"),
        "hospital_name": bill_data.get("hospital_name"),
        "policy_number": bill_data.get("policy_number"),
        "treatment": bill_data.get("treatment"),
        "total_bill": bill_data.get("total_bill_amount"),
        "covered_amount": verification["covered_amount"],
        "non_covered_amount": verification["non_covered_amount"],
        "verification_status": verification["verification_status"],
        "covered_items": verification["covered_items"],
        "rejected_items": verification["rejected_items"]
    }

    return result

uploaded = files.upload()

bill_filename = list(uploaded.keys())[0]

print("Uploaded:", bill_filename)

result = analyze_claim(
    claim_id="CLM1001",
    age=45,
    hospitalization_days=7,
    medical_expenses=85000,
    previous_claims=2,
    bill_filename=bill_filename
)

print(json.dumps(result, indent=4))

print("=" * 60)
print("       INSURANCE CLAIM ANALYSIS REPORT")
print("=" * 60)

print("Claim ID:", result["claim_id"])
print("Patient:", result["patient_name"])
print("Hospital:", result["hospital_name"])
print("Policy Number:", result["policy_number"])
print("Treatment:", result["treatment"])

print("-" * 60)

print("Claim Severity:", result["claim_severity"])
print("Total Bill: ₹", result["total_bill"])
print("Covered Amount: ₹", result["covered_amount"])
print("Non-Covered Amount: ₹", result["non_covered_amount"])
print("Verification:", result["verification_status"])

print("-" * 60)

print("Covered Items:")

for item in result["covered_items"]:
    print("✓", item)

print("\nRejected / Non-Covered Items:")

for item in result["rejected_items"]:
    print("✗", item)

print("=" * 60)

with open(
    "final_claim_report.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        result,
        file,
        indent=4
    )

print("Final report saved as final_claim_report.json")

amounts = [
    result["covered_amount"],
    result["non_covered_amount"]
]

labels = [
    "Covered",
    "Non-Covered"
]

plt.figure(figsize=(7,5))

plt.bar(labels, amounts)

plt.title("Insurance Claim Coverage")
plt.ylabel("Amount (₹)")

plt.show()

import gradio as gr

def gradio_analysis(
    claim_id,
    age,
    hospitalization_days,
    medical_expenses,
    previous_claims,
    bill_file
):

    if bill_file is None:
        return "Please upload a medical bill."

    try:

        result = analyze_claim(
            claim_id,
            int(age),
            int(hospitalization_days),
            float(medical_expenses),
            int(previous_claims),
            bill_file
        )

        return json.dumps(
            result,
            indent=4
        )

    except Exception as e:

        return "Error: " + str(e)

Idemo = gr.Interface(
    fn=gradio_analysis,

    inputs=[
        gr.Textbox(label="Claim ID"),
        gr.Number(label="Age"),
        gr.Number(label="Hospitalization Days"),
        gr.Number(label="Medical Expenses"),
        gr.Number(label="Previous Claims"),

        gr.File(
            label="Upload Medical Bill",
            file_types=[
                ".pdf",
                ".png",
                ".jpg",
                ".jpeg",
                ".txt"
            ],
            type="filepath"
        )
    ],

    outputs=gr.Textbox(
        label="Final Claim Analysis",
        lines=20
    ),

    title="Insurance Claim Severity & Medical Bill Verification",

    description="""
    Predict claim severity using Machine Learning,
    extract medical bill information using an LLM,
    and verify expenses against predefined policy rules.
    """
)

demo.launch(share=True)